In [ ]:
import pandas as pd
import json
from tqdm.notebook import tqdm
import plotly.express as px
from datetime import datetime

In [10]:
name = 'sensor_log_2026-02-21_08-46-57'

In [11]:
imu_data = {
    'accelerometer': [],
    'gyroscope': [],
    'magnetometer': []
}
gnss_measurments = []
with open(f'./data/{name}.jsonl', 'r') as f:
    for line in tqdm(f.readlines()):
        line_data = json.loads(line)
        if line_data['type'] == 'sensor_measurement':
            if line_data['sensor'] == 'accelerometer':
                imu_data['accelerometer'].append(line_data)
            elif line_data['sensor'] == 'gyroscope':
                imu_data['gyroscope'].append(line_data)
            elif line_data['sensor'] == 'magnetometer':
                imu_data['magnetometer'].append(line_data)
        elif line_data['type'] == 'gnss_measurement':
            for m in line_data['measurements']:
                gnss_measurments.append(m | {'timestamp': datetime.fromtimestamp(line_data['timestamp'] / 1e3)})
        elif line_data['type'] == 'sensor_metadata':
            ...
        elif line_data['type'] == 'nmea_message':
            ...
        elif line_data['type'] == 'gnss_navigation_message':
            ...
        else:
            raise ValueError(f"Unknown type {line_data['type']}")

  0%|          | 0/170074 [00:00<?, ?it/s]

In [12]:
gnss_data = pd.DataFrame.from_records(gnss_measurments)
gnss_data

,accumulatedDeltaRangeMeters,accumulatedDeltaRangeState,accumulatedDeltaRangeUncertaintyMeters,carrierFrequencyHz,carrierPhase,cn0DbHz,constellationType,pseudorangeRateMetersPerSecond,pseudorangeRateUncertaintyMetersPerSecond,receivedSvTimeNanos,receivedSvTimeUncertaintyNanos,state,svid,timeOffsetNanos,timestamp
0,0.0,16,0.0,1.605375e+09,0.0,5.000000,3,1300.687510,0.14999,598064,396,17,8,0.0,2026-02-21 08:47:11.813
1,0.0,16,0.0,1.605375e+09,0.0,5.000000,3,1294.650552,0.14999,593704,396,17,8,0.0,2026-02-21 08:47:12.782
2,0.0,16,0.0,1.575420e+09,0.0,37.615002,1,1215.367221,0.14999,568051621949788,51,16431,4,0.0,2026-02-21 08:47:13.780
3,0.0,16,0.0,1.575420e+09,0.0,28.850000,1,362.193994,0.14999,568051624301653,316,16431,8,0.0,2026-02-21 08:47:13.780
4,0.0,16,0.0,1.605375e+09,0.0,5.000000,3,1290.761047,0.14999,589311,74,17,8,0.0,2026-02-21 08:47:13.780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7875,0.0,16,0.0,1.575420e+09,0.0,30.100000,6,630.969776,0.14999,568355907252354,74,16431,15,0.0,2026-02-21 08:52:18.113
7876,0.0,16,0.0,1.575420e+09,0.0,29.600000,6,739.684515,0.14999,743795,74,17,28,0.0,2026-02-21 08:52:18.113
7877,0.0,16,0.0,1.575420e+09,0.0,39.500000,6,1302.160555,0.14999,568355911604221,42,16431,27,0.0,2026-02-21 08:52:18.113
7878,0.0,16,0.0,1.575420e+09,0.0,28.100000,6,1420.190670,0.14999,568355913374908,235,16431,29,0.0,2026-02-21 08:52:18.113


In [13]:
gnss_data.accumulatedDeltaRangeState.value_counts() / len(gnss_data)

accumulatedDeltaRangeState
16    0.769670
25    0.177157
17    0.053046
21    0.000127
Name: count, dtype: float64

In [14]:
(gnss_data.accumulatedDeltaRangeMeters == 0).value_counts() / len(gnss_data)

accumulatedDeltaRangeMeters
True     0.768528
False    0.231472
Name: count, dtype: float64

In [15]:
phase_counts = gnss_data.groupby('timestamp').apply(lambda x: (x.accumulatedDeltaRangeMeters != 0).sum(), include_groups = False)

In [16]:
px.line(phase_counts.iloc[200:])

In [17]:
phase_counts.iloc[200:].value_counts() / len(phase_counts.iloc[200:])

5     0.186916
4     0.140187
6     0.130841
8     0.093458
7     0.093458
3     0.084112
2     0.084112
9     0.084112
1     0.037383
10    0.037383
0     0.018692
12    0.009346
Name: count, dtype: float64